<a href="https://colab.research.google.com/github/JEAN-hub-lab/APPENDING-JOINING/blob/main/Chapter_8_Preparing_Column_for_Modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 500

df1 = pd.DataFrame({
    "Area": rng.normal(120, 40, n).clip(20, 400),
    "Bedrooms": rng.integers(1, 6, n),  # ไม่มี 0
    "Age": rng.integers(0, 40, n).astype(float),
    "Location": rng.choice(["city", "City", "suburb", "SUBURB", "rural"], n),
    "Condition": rng.choice(["new", "good", "old"], n),
    "Description": rng.choice(["near school", "near mall", "park view", ""], n, p=[0.30,0.20,0.20,0.30]),
})

# Price สุ่ม (ไม่สัมพันธ์กับ feature)
df1["Price"] = rng.normal(300_000, 80_000, n).clip(50_000, 800_000)

df1.loc[rng.choice(df1.index, int(0.15*n), replace=False), "Age"] = np.nan
df1.loc[rng.choice(df1.index, 12, replace=False), "Price"] *= 3.0

df1.head()


,Area,Bedrooms,Age,Location,Condition,Description,Price
0,132.188683,3,NaN,SUBURB,old,park view,148546.307101
1,78.400636,2,22.0,SUBURB,good,,317381.852658
2,150.018048,4,9.0,City,old,,426006.626209
3,157.622589,1,23.0,rural,good,,339093.481957
4,41.958592,2,27.0,suburb,old,near school,313650.006283


Step 1 — Data Audit (ตรวจ missing + ภาพรวม)

In [ ]:
df1.info()



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Area         500 non-null    float64
 1   Bedrooms     500 non-null    int64  
 2   Age          425 non-null    float64
 3   Location     500 non-null    object 
 4   Condition    500 non-null    object 
 5   Description  500 non-null    object 
 6   Price        500 non-null    float64
dtypes: float64(3), int64(1), object(3)
memory usage: 27.5+ KB


In [ ]:
df1.isna().sum().sort_values(ascending=False)



,0
Age,75
Area,0
Bedrooms,0
Location,0
Condition,0
Description,0
Price,0


In [ ]:
df1.describe(include="all")

,Area,Bedrooms,Age,Location,Condition,Description,Price
count,500.000000,500.000000,425.000000,500,500,500,5.000000e+02
unique,NaN,NaN,NaN,5,3,4,NaN
top,NaN,NaN,NaN,City,good,,NaN
freq,NaN,NaN,NaN,109,178,162,NaN
mean,119.480287,3.014000,19.454118,NaN,NaN,NaN,3.146590e+05
std,38.383333,1.404903,11.500959,NaN,NaN,NaN,1.321338e+05
min,20.000000,1.000000,0.000000,NaN,NaN,NaN,5.489761e+04
25%,93.323306,2.000000,10.000000,NaN,NaN,NaN,2.446085e+05
50%,120.128469,3.000000,20.000000,NaN,NaN,NaN,3.044974e+05
75%,143.495733,4.000000,29.000000,NaN,NaN,NaN,3.528412e+05


In [ ]:
#Age มี outlier หรือไม่ หาค่าอย่างไร
#Price มี outlier หรือไม่ หาค่าอย่างไร

In [ ]:
Q1 = df1["Age"].quantile(0.25)
Q3 = df1["Age"].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5*IQR
upper = Q3 + 1.5*IQR

df1[(df1["Age"] < lower) | (df1["Age"] > upper)]


,Area,Bedrooms,Age,Location,Condition,Description,Price


In [ ]:
Q1 = df1["Price"].quantile(0.25)
Q3 = df1["Price"].quantile(0.75)
IQR = Q3 - Q1


lower = Q1 - 1.5*IQR
upper = Q3 + 1.5*IQR
print(lower,upper)

df1[(df1["Price"] < lower) | (df1["Price"] > upper)]

82259.548493204 515190.1929777263


,Area,Bedrooms,Age,Location,Condition,Description,Price
15,85.628301,4,NaN,SUBURB,old,,1.030878e+06
25,105.914658,4,24.0,rural,good,near school,7.401893e+05
49,122.703163,3,13.0,City,good,near mall,5.467650e+05
70,68.972547,1,26.0,rural,new,near school,1.058506e+06
77,126.341588,3,39.0,SUBURB,old,,5.285951e+05
115,98.820292,5,34.0,SUBURB,new,near mall,1.051057e+06
151,78.573850,4,15.0,City,good,,7.742945e+05
166,157.262921,2,NaN,City,old,park view,5.163188e+05
175,126.903517,4,15.0,rural,old,near school,1.075558e+06
190,188.946629,5,27.0,rural,old,near mall,5.250119e+05


ความเบ้วัดด้วย skewness (>1 มาก, >2 รุนแรง)

outlier วัดด้วย IQR ratio (>5% ถือว่าเยอะ)


In [ ]:
import pandas as pd

# 1) เลือกเฉพาะคอลัมน์ตัวเลข
num_cols = df1.select_dtypes(include=["number"]).columns

# 2) ฟังก์ชันหา % outlier ด้วย IQR rule
def outlier_percent_iqr(s: pd.Series) -> float:
    s = s.dropna()
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return ((s < lower) | (s > upper)).mean() * 100

# 3) สรุป skewness และ %outlier เป็นตารางเดียว
summary = pd.DataFrame({
    "skewness": df1[num_cols].skew(numeric_only=True),
    "outlier_% (IQR)": df1[num_cols].apply(outlier_percent_iqr),
    "missing_%": df1[num_cols].isna().mean() * 100
}).sort_values(by="outlier_% (IQR)", ascending=False)

summary


,skewness,outlier_% (IQR),missing_%
Price,3.376374,3.6,0.0
Area,0.096267,0.8,0.0
Bedrooms,0.005538,0.0,0.0
Age,0.026845,0.0,15.0


Step 2 — Split ก่อนเสมอ (กัน leakage) + log transform to target variable

In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np

X1 = df1.drop(columns=["Price"])
y1_log = np.log(df1["Price"])

X1_train, X1_test, y1_train, y1_test = train_test_split(
    X1, y1_log, test_size=0.2, random_state=42
)

X1_train.shape, X1_test.shape


((400, 6), (100, 6))

Step 3 — (optional) วัด skewness / %outlier เฉพาะ TRAIN

In [ ]:
num_cols = X1_train.select_dtypes(include=["number"]).columns

def outlier_percent_iqr(s: pd.Series) -> float:
    s = s.dropna()
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return ((s < lower) | (s > upper)).mean() * 100

summary_train = pd.DataFrame({
    "skewness": X1_train[num_cols].skew(numeric_only=True),
    "outlier_% (IQR)": X1_train[num_cols].apply(outlier_percent_iqr),
    "missing_%": X1_train[num_cols].isna().mean() * 100
})

summary_train


,skewness,outlier_% (IQR),missing_%
Area,0.128270,1.25,0.0
Bedrooms,-0.006991,0.00,0.0
Age,-0.000461,0.00,16.0


Step 4 — สร้าง Transformer สำหรับแปลง Text → Numeric (TextFlags)

In [ ]:
# ถ้า Description เป็น:
# "Near school and park" → near_school=1, near_mall=0
# "near mall" → near_school=0, near_mall=1
# "" หรือ None → 0, 0

from sklearn.base import BaseEstimator, TransformerMixin

class TextFlags(BaseEstimator, TransformerMixin):
    def __init__(self, col: str):
        self.col = col
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        s = X[self.col].fillna("").astype(str)
        return pd.DataFrame({
            "near_school": s.str.contains("school", case=False, na=False).astype(int),
            "near_mall":   s.str.contains("mall",   case=False, na=False).astype(int),
        }, index=X.index)
    def get_feature_names_out(self, input_features=None):
        return ["near_school", "near_mall"]


Step 5 — สร้าง Preprocessing ด้วย ColumnTransformer (Numeric/Cat/Text)+Pipeline

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression

num_cols = ["Area", "Bedrooms", "Age"]
cat_cols = ["Location", "Condition"]
txt_col = "Description"

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocess_1_base = ColumnTransformer([
    ("num", numeric_pipe, num_cols),
    ("cat", categorical_pipe, cat_cols),
    ("txt", TextFlags(txt_col), [txt_col]),
], remainder="drop")

pipe_1_base = Pipeline([
    ("prep", preprocess_1_base),
    ("model", LinearRegression())
])


In [ ]:
pipe_1_base

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Area', 'Bedrooms', 'Age']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Location', 'Condition']),
                                                 ('txt',
                                                  TextFlags(col='Description'),
                                                  ['Description'])])),
                ('model', LinearRegression())])

Step 6 — สร้าง Pipeline บน Dataset 1 เสร็จแล้ว Train+Evaluate Baseline (RMSE/R² บน “ราคาจริง”)

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

pipe_1_base.fit(X1_train, y1_train)
pred1_log_base = pipe_1_base.predict(X1_test)



real1_price = np.exp(y1_test)
pred1_price_base = np.exp(pred1_log_base)

rmse1_base = np.sqrt(mean_squared_error(real1_price, pred1_price_base))
r2_1_base = r2_score(real1_price, pred1_price_base)

rmse1_base, r2_1_base

(np.float64(133094.54390126883), -0.036283035078015)

In [ ]:
# X_trans = pipe_1_base.named_steps["prep"].transform(X1_test)

# # เลือกดู 5 แถวแรก และ 20 feature แรก
# X_trans[:5, :20]

# feature_names = pipe_1_base.named_steps["prep"].get_feature_names_out()

# X_trans_df = pd.DataFrame(
#     X_trans[:5].toarray() if hasattr(X_trans, "toarray") else X_trans[:5],
#     columns=feature_names
# )

# X_trans_df.iloc[:, :25]   # แสดง 25 คอลัมน์แรก


Step 8 — เพิ่ม Feature (Area_per_bedroom) เฉพาะ train/test + rebuild pipeline

In [ ]:
X1_train2 = X1_train.copy()
X1_test2  = X1_test.copy()

X1_train2["Area_per_bedroom"] = X1_train2["Area"] / X1_train2["Bedrooms"].replace(0, np.nan)
X1_test2["Area_per_bedroom"]  = X1_test2["Area"]  / X1_test2["Bedrooms"].replace(0, np.nan)

num_cols2 = ["Area", "Bedrooms", "Age", "Area_per_bedroom"]

preprocess_1_fe = ColumnTransformer([
    ("num", numeric_pipe, num_cols2),
    ("cat", categorical_pipe, cat_cols),
    ("txt", TextFlags(txt_col), [txt_col]),
], remainder="drop")

pipe_1_fe = Pipeline([
    ("prep", preprocess_1_fe),
    ("model", LinearRegression())
])


In [ ]:
pipe_1_fe

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Area', 'Bedrooms', 'Age',
                                                   'Area_per_bedroom']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Location', 'Condition']),
                                                 ('txt',
                                                  TextFlags(col='Description'),
                                                  ['Description'])])),
                ('model', LinearRegression())])

Step 8.1 Fit + Predict + Evaluate (With Area_per_bedroom on Dataset 1)

In [ ]:
pipe_1_fe.fit(X1_train2, y1_train)
pred1_log_fe = pipe_1_fe.predict(X1_test2)

pred1_price_fe = np.exp(pred1_log_fe)

rmse1_fe = np.sqrt(mean_squared_error(real1_price, pred1_price_fe))
r2_1_fe = r2_score(real1_price, pred1_price_fe)

rmse1_fe, r2_1_fe


(np.float64(132904.20195180192), -0.03332112420638467)

Step 8.2 สรุปผล Dataset 1

In [ ]:
pd.DataFrame({
    "Dataset": ["Dataset 1 (No signal)"]*2,
    "Model": ["Baseline", "With Area_per_bedroom"],
    "RMSE": [rmse1_base, rmse1_fe],
    "R2": [r2_1_base, r2_1_fe],
    "RMSE_change (neg=better)": [0.0, rmse1_fe - rmse1_base],
})


,Dataset,Model,RMSE,R2,RMSE_change (neg=better)
0,Dataset 1 (No signal),Baseline,133094.543901,-0.036283,0.000000
1,Dataset 1 (No signal),With Area_per_bedroom,132904.201952,-0.033321,-190.341949


# DataSet 2 ให้ Price ขึ้นกับ Area_per_bedroom

Step 2.1 — สร้าง Dataset 2 โดย “คัดลอก features เดิมจาก df1” แล้วสร้าง Price ใหม่

- จุดสำคัญ: Features เหมือน Dataset 1 ทุกอย่าง แต่ Price ถูกสร้างให้สัมพันธ์กับ Area_per_bedroom

In [ ]:
df2 = df1.drop(columns=["Price"]).copy()

loc_clean = df2["Location"].astype(str).str.strip().str.lower()
cond_clean = df2["Condition"].astype(str).str.strip().str.lower()

loc_premium = loc_clean.map({"city": 60_000, "suburb": 20_000, "rural": -10_000}).fillna(0)
cond_premium = cond_clean.map({"new": 40_000, "good": 10_000, "old": -20_000}).fillna(0)

near_school = df2["Description"].astype(str).str.contains("school", case=False, na=False).astype(int)
near_mall   = df2["Description"].astype(str).str.contains("mall",   case=False, na=False).astype(int)

area_per_bedroom = df2["Area"] / df2["Bedrooms"].replace(0, np.nan)


noise = rng.normal(0, 10_000, len(df2))

df2["Price"] = (
    90_000
    + 6_500 * area_per_bedroom          # ✅ เพิ่ม signal นิดหน่อย
    - 900 * df2["Age"].fillna(df2["Age"].median())
    + loc_premium + cond_premium
    + 10_000 * near_school + 8_000 * near_mall
    + noise
).clip(50_000, None)


df2.loc[rng.choice(df2.index, 6, replace=False), "Price"] *= 2.0

df2.head()


,Area,Bedrooms,Age,Location,Condition,Description,Price
0,132.188683,3,NaN,SUBURB,old,park view,3.668045e+05
1,78.400636,2,22.0,SUBURB,good,,3.693521e+05
2,150.018048,4,9.0,City,old,,3.805806e+05
3,157.622589,1,23.0,rural,good,,1.097436e+06
4,41.958592,2,27.0,suburb,old,near school,2.125260e+05


Step 2.2 — Audit Dataset 2

In [ ]:
df2.info()



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Area         500 non-null    float64
 1   Bedrooms     500 non-null    int64  
 2   Age          425 non-null    float64
 3   Location     500 non-null    object 
 4   Condition    500 non-null    object 
 5   Description  500 non-null    object 
 6   Price        500 non-null    float64
dtypes: float64(3), int64(1), object(3)
memory usage: 27.5+ KB


In [ ]:
df2.isna().sum().sort_values(ascending=False)


,0
Age,75
Area,0
Bedrooms,0
Location,0
Condition,0
Description,0
Price,0


In [ ]:
df2.describe(include="all")

,Area,Bedrooms,Age,Location,Condition,Description,Price
count,500.000000,500.000000,425.000000,500,500,500,5.000000e+02
unique,NaN,NaN,NaN,5,3,4,NaN
top,NaN,NaN,NaN,City,good,,NaN
freq,NaN,NaN,NaN,109,178,162,NaN
mean,119.480287,3.014000,19.454118,NaN,NaN,NaN,4.693768e+05
std,38.383333,1.404903,11.500959,NaN,NaN,NaN,2.619512e+05
min,20.000000,1.000000,0.000000,NaN,NaN,NaN,1.132289e+05
25%,93.323306,2.000000,10.000000,NaN,NaN,NaN,2.904546e+05
50%,120.128469,3.000000,20.000000,NaN,NaN,NaN,3.899529e+05
75%,143.495733,4.000000,29.000000,NaN,NaN,NaN,5.434839e+05


In [ ]:
import pandas as pd

# 1) เลือกเฉพาะคอลัมน์ตัวเลข
num_cols = df2.select_dtypes(include=["number"]).columns

# 2) ฟังก์ชันหา % outlier ด้วย IQR rule
def outlier_percent_iqr(s: pd.Series) -> float:
    s = s.dropna()
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return ((s < lower) | (s > upper)).mean() * 100

# 3) สรุป skewness และ %outlier เป็นตารางเดียว
summary = pd.DataFrame({
    "skewness": df2[num_cols].skew(numeric_only=True),
    "outlier_% (IQR)": df2[num_cols].apply(outlier_percent_iqr),
    "missing_%": df1[num_cols].isna().mean() * 100
}).sort_values(by="outlier_% (IQR)", ascending=False)

summary

,skewness,outlier_% (IQR),missing_%
Price,1.786176,8.4,0.0
Area,0.096267,0.8,0.0
Bedrooms,0.005538,0.0,0.0
Age,0.026845,0.0,15.0


Step 2.3 — Split + log target (Dataset 2)

In [ ]:
X2 = df2.drop(columns=["Price"])
y2_log = np.log(df2["Price"])

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2_log, test_size=0.2, random_state=42
)

X2_train.shape, X2_test.shape


((400, 6), (100, 6))

Step 2.4 — Baseline pipeline (เหมือนเดิม) + Fit/Evaluate

In [ ]:
pipe_2_base = Pipeline([
    ("prep", preprocess_1_base),   # ใช้ preprocess เดิมได้ เพราะ schema เดิม
    ("model", LinearRegression())
])


pipe_2_base.fit(X2_train, y2_train)
pred2_log_base = pipe_2_base.predict(X2_test)

real2_price = np.exp(y2_test)
pred2_price_base = np.exp(pred2_log_base)

rmse2_base = np.sqrt(mean_squared_error(real2_price, pred2_price_base))
r2_2_base = r2_score(real2_price, pred2_price_base)

rmse2_base, r2_2_base


(np.float64(99602.62820024243), 0.8648376946575314)

In [ ]:
pipe_2_base

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Area', 'Bedrooms', 'Age']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Location', 'Condition']),
                                                 ('txt',
                                                  TextFlags(col='Description'),
                                                  ['Description'])])),
                ('model', LinearRegression())])

Step 2.5 — เพิ่ม Area_per_bedroom (Dataset 2) + rebuild + Fit/Evaluate

In [ ]:
X2_train2 = X2_train.copy()
X2_test2  = X2_test.copy()

X2_train2["Area_per_bedroom"] = X2_train2["Area"] / X2_train2["Bedrooms"].replace(0, np.nan)
X2_test2["Area_per_bedroom"]  = X2_test2["Area"]  / X2_test2["Bedrooms"].replace(0, np.nan)



pipe_2_fe = Pipeline([
    ("prep", preprocess_1_fe),     # preprocess ที่มี num_cols2
    ("model", LinearRegression())
])

pipe_2_fe.fit(X2_train2, y2_train)
pred2_log_fe = pipe_2_fe.predict(X2_test2)

pred2_price_fe = np.exp(pred2_log_fe) #คือการทำ “**ย้อนกลับจาก log**” ไปเป็นราคาจริง

rmse2_fe = np.sqrt(mean_squared_error(real2_price, pred2_price_fe))
r2_2_fe = r2_score(real2_price, pred2_price_fe)

rmse2_fe, r2_2_fe


(np.float64(78548.45369773217), 0.9159400216213852)

Step 2.6 — สรุปผล Dataset 2

In [ ]:
pd.DataFrame({
    "Dataset": ["Dataset 2 (Injected signal)"]*2,
    "Model": ["Baseline", "With Area_per_bedroom"],
    "RMSE": [rmse2_base, rmse2_fe],
    "R2": [r2_2_base, r2_2_fe],
    "RMSE_change (neg=better)": [0.0, rmse2_fe - rmse2_base],
})


,Dataset,Model,RMSE,R2,RMSE_change (neg=better)
0,Dataset 2 (Injected signal),Baseline,99602.628200,0.864838,0.000000
1,Dataset 2 (Injected signal),With Area_per_bedroom,78548.453698,0.915940,-21054.174503


Step สุดท้าย — ตารางรวม 2 Dataset

In [ ]:
pd.DataFrame([
    ["Dataset 1 (No signal)", "Baseline", rmse1_base, r2_1_base],
    ["Dataset 1 (No signal)", "With Area_per_bedroom", rmse1_fe, r2_1_fe],
    ["Dataset 2 (Injected signal)", "Baseline", rmse2_base, r2_2_base],
    ["Dataset 2 (Injected signal)", "With Area_per_bedroom", rmse2_fe, r2_2_fe],
], columns=["Dataset", "Model", "RMSE", "R2"])


,Dataset,Model,RMSE,R2
0,Dataset 1 (No signal),Baseline,133094.543901,-0.036283
1,Dataset 1 (No signal),With Area_per_bedroom,132904.201952,-0.033321
2,Dataset 2 (Injected signal),Baseline,99602.628200,0.864838
3,Dataset 2 (Injected signal),With Area_per_bedroom,78548.453698,0.915940


ดูผลลัพธ์ที่ได้จาก Test set

In [ ]:
import numpy as np
import pandas as pd

# 1) predict (เป็น log-price)
pred_log = pipe_2_fe.predict(X2_test2)

# 2) แปลงกลับเป็นราคาจริง
true_price = np.exp(y2_test)
pred_price = np.exp(pred_log)

# 3) สร้างตารางสรุป
result_df = X2_test2.copy()
result_df["true_price"] = true_price.values
result_df["pred_price"] = pred_price
result_df["abs_error"] = (result_df["true_price"] - result_df["pred_price"]).abs()
result_df["pct_error_%"] = (result_df["abs_error"] / result_df["true_price"] * 100)

# 4) แสดง 20 แถวแรก
result_df.head(20)


,Area,Bedrooms,Age,Location,Condition,Description,Area_per_bedroom,true_price,pred_price,abs_error,pct_error_%
361,135.335755,2,9.0,suburb,good,near mall,67.667877,5.495436e+05,5.501876e+05,644.001543,0.117188
73,139.886430,3,6.0,City,old,,46.628810,4.285888e+05,4.242126e+05,4376.271658,1.021089
374,142.164660,4,11.0,SUBURB,good,near school,35.541165,3.598143e+05,3.403390e+05,19475.316895,5.412602
155,190.717197,5,5.0,rural,good,,38.143439,3.326847e+05,3.251333e+05,7551.369040,2.269828
104,82.655293,3,13.0,city,good,,27.551764,3.203856e+05,3.256779e+05,5292.229100,1.651831
394,148.078145,3,14.0,SUBURB,old,near school,49.359382,4.157846e+05,4.040502e+05,11734.376488,2.822225
377,39.859108,3,21.0,suburb,good,near school,13.286369,1.899092e+05,2.305931e+05,40683.890798,21.422811
124,153.404450,1,30.0,City,new,near mall,153.404450,1.165913e+06,1.463835e+06,297922.291400,25.552704
68,154.319035,4,31.0,City,old,near school,38.579759,3.613171e+05,3.543893e+05,6927.843144,1.917386
450,114.754499,1,11.0,rural,new,,114.754499,8.666075e+05,8.172104e+05,49397.130230,5.700058


ดู “เคสที่โมเดลพลาดมากที่สุด”

In [ ]:
result_df.sort_values("abs_error", ascending=False).head(20)


,Area,Bedrooms,Age,Location,Condition,Description,Area_per_bedroom,true_price,pred_price,abs_error,pct_error_%
268,122.126214,3,9.0,City,good,near school,40.708738,8.506883e+05,4.117068e+05,438981.477941,51.603096
124,153.404450,1,30.0,City,new,near mall,153.404450,1.165913e+06,1.463835e+06,297922.291400,25.552704
278,167.645718,1,12.0,city,good,park view,167.645718,1.237822e+06,1.485289e+06,247467.143087,19.992146
323,150.034760,1,16.0,city,new,near school,150.034760,1.163882e+06,1.398866e+06,234984.422220,20.189716
495,157.421885,1,1.0,suburb,new,near school,157.421885,1.173307e+06,1.401102e+06,227795.512631,19.414828
362,159.992970,1,13.0,City,good,,159.992970,1.197376e+06,1.372229e+06,174853.532742,14.603062
324,192.825846,2,4.0,City,new,park view,96.412923,8.220220e+05,9.728025e+05,150780.527429,18.342639
491,134.661256,1,27.0,city,new,near school,134.661256,1.059888e+06,1.167869e+06,107981.027056,10.187966
79,107.626138,1,NaN,rural,good,,107.626138,7.682778e+05,6.618104e+05,106467.391603,13.857929
193,178.457772,2,31.0,SUBURB,new,near mall,89.228886,7.140027e+05,8.047021e+05,90699.416605,12.702951
